In [ ]:
import random
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import os.path
import pandas as pd
from sklearn import preprocessing
import os

# User inputs

In [ ]:
sample_size = 512 #enter the size to which events will be up/downsampled

# designed for data in the following format:
# x[0] ,y[1] ,z[2] ,time[3], Amplitude[4]
TRACK_CLASS = False
dimension = 4 # desired dimension of data to be input
ISOTOPE = 'O16'

In [ ]:
data = np.load('../voxel_data/' + ISOTOPE + '_w_event_keys.npy')
event_lens = np.load('../voxel_data/' + ISOTOPE + '_event_lens.npy')
data[56437,0,-1] = 56437
assert data.shape == (event_lens.size, np.max(event_lens), 6), 'Array has incorrect shape'
assert len(np.unique(data[:,:,5])) == event_lens.size, 'Array has incorrect Event_ids'

# Filter the Data

In [ ]:
def filter_point_clouds(data):
    
    # Extract only the x, y, z coordinates, and the charge (columns 0, 1, 2, 4)
    filtered_data = data[..., [0, 1, 2, 4]]
    return filtered_data

# Apply the filter function to the data
filtered_data = filter_point_clouds(data)

# Apply additional filtering based on the charge value
filtered_data[:,:,0] = np.where(filtered_data[:,:,3] < 90, 0, filtered_data[:,:,0])
filtered_data[:,:,1] = np.where(filtered_data[:,:,3] < 90, 0, filtered_data[:,:,1])
filtered_data[:,:,2] = np.where(filtered_data[:,:,3] < 90, 0, filtered_data[:,:,2])
filtered_data[:,:,3] = np.where(filtered_data[:,:,3] < 90, 0, filtered_data[:,:,3])

# Determine threshold for minimum number of non-zero points
min_points_threshold = 70

# Filter out events with too few non-zero points
filtered_events = []
valid_event_indices = []
for i in range(filtered_data.shape[0]):
    if np.count_nonzero(filtered_data[i, :, 0]) >= min_points_threshold:
        filtered_events.append(filtered_data[i])
        valid_event_indices.append(i)

filtered_data = np.array(filtered_events)

# Print the final filtered data for verification
print('Final Filtered Shape:', filtered_data.shape)
print(len(valid_event_indices))

# Random sample from NumPy array

In [ ]:
new_array_name = ISOTOPE + '_size' + str(sample_size) + '_sampled'
new_data = np.zeros((filtered_data.shape[0], sample_size, 6), float)
# Using tqdm with enumerate for progress indication
for idx, original_idx in tqdm.tqdm(enumerate(valid_event_indices), total=len(valid_event_indices)):
    # Filter out zero values using idx instead of original_idx
    non_zero_points = filtered_data[idx][filtered_data[idx, :, 0] != 0]
    non_zero_len = non_zero_points.shape[0]

    if original_idx == 56437:  # skipping the one empty event
        new_data[idx, 0, 5] = 56437
        continue

    if non_zero_len > sample_size:
        random_points = np.random.choice(non_zero_len, sample_size, replace=False)  # choosing the random instances to sample
        for count, r in enumerate(random_points):
            new_data[idx, count, :3] = non_zero_points[r, :3]  # Only use the filtered x, y, z
            new_data[idx, count, 3] = data[original_idx, r, 3]  # adding time from original data
            new_data[idx, count, 4] = non_zero_points[r, 3]  # use amplitude (charge) from filtered data
            new_data[idx, count, 5:] = data[original_idx, r, 5:]  # Add the remaining columns (event index) from the original data
    else:
        new_data[idx, :non_zero_len, :3] = non_zero_points[:, :3]  # Only use the filtered x, y, z
        new_data[idx, :non_zero_len, 3] = data[original_idx, :non_zero_len, 3]  # adding time from original data
        new_data[idx, :non_zero_len, 4] = non_zero_points[:, 3]  # use amplitude (charge) from filtered data
        new_data[idx, :non_zero_len, 5:] = data[original_idx, :non_zero_len, 5:]  # Add the remaining columns (event index) from the original data
        need = sample_size - non_zero_len
        random_points = np.random.choice(non_zero_len, need, replace=True if need > non_zero_len else False)
        for count, r in enumerate(random_points, start=non_zero_len):
            new_data[idx, count, :3] = non_zero_points[r, :3]  # Only use the filtered x, y, z
            new_data[idx, count, 3] = data[original_idx, r, 3]  # adding time from original data
            new_data[idx, count, 4] = non_zero_points[r, 3]  # use amplitude (charge) from filtered data
            new_data[idx, count, 5:] = data[original_idx, r, 5:]  # Add the remaining columns (event index) from the original data
    new_data[idx, 0, 5] = data[original_idx, 0, 5]  # saving the event index

# Verify if there are still any zero values in new_data
print("Final check of new_data for zeros")
print("Number of zero values in x, y, z, charge columns:", np.count_nonzero(new_data[:, :, :4] == 0))
print("Indices of zero values:", np.where(new_data[:, :, :4] == 0))

# If there are still zeros, print a few samples where zeros are present
zero_indices = np.where(new_data[:, :, :4] == 0)
if len(zero_indices[0]) > 0:
    for i in range(min(5, len(zero_indices[0]))):
        print(f"Zero found at index {zero_indices[0][i]}, event {zero_indices[1][i]}, column {zero_indices[2][i]}")

assert np.all(new_data[:, :, :4] != 0), 'new_data contains zero values in the x, y, z, or charge columns'

np.save('../voxel_data/' + new_array_name, new_data)

assert new_data.shape == (filtered_data.shape[0], sample_size, 6), 'Array has incorrect shape'
assert len(np.unique(new_data[:, :, 5])) == filtered_data.shape[0], 'Array has incorrect number of events'

In [ ]:
def print_column_values(file_path): 
    """Print values of the first column from the loaded data"""
    event_data = np.load(file_path)
    for n in range(10): #increase readability by just printing the first column of the first 10 events
    # for n in range(len(event_data)):
        column_values = event_data[n, 0, :]
        print(f'Values of first column:\n{column_values}')

print_column_values('../voxel_data/O16_size512_sampled.npy')

In [ ]:
size = len(new_data)
dataset = np.zeros((size, sample_size, dimension + 3), float)
count = 0

for i in range(size):
    dataset[count,:,:3] = new_data[count,:,:3] # x, y, z
    if dimension == 4: 
        dataset[count,:,3] = new_data[count,:,4] # charge
    dataset[count,0,-3] = new_data[count,0,5] # event index
    # [count,0,-2] -- number of tracks, leave as 0
    dataset[count,0,-1] = np.count_nonzero(new_data[count,:,0])# length of event 
    count += 1

np.save('../voxel_data/' + ISOTOPE + '_size' + str(sample_size), dataset)
print(len(dataset))

# Scaling

In [ ]:
# values correspond to the x,y,z,charge index
values = [0,1,2,3] 
means_and_stds = []
min_max_scaler = preprocessing.MinMaxScaler(feature_range=(-1, 1))
dataset[:,:,3] = np.log(dataset[:,:,3]) # log scale charge
# standard scaling 
for n in values:
    mean = np.mean(dataset[:,:,n])
    std = np.std(dataset[:,:,n])
    means_and_stds.append([mean,std])
    dataset[:,:,n] = (dataset[:,:,n] - mean) / std
dataset[:,0,-1] = np.log(dataset[:,0,-1])
dataset[:,0,-1] = min_max_scaler.fit_transform(dataset[:,0,-1].reshape(-1, 1)).reshape(1,-1)

assert np.sum(np.isnan(dataset)) == 0, 'NaNs in dataset'
assert np.sum(np.isinf(dataset)) == 0, 'Infinities in dataset'

# Split into training, testing, and validation sets¶
Performs a 60-20-20 split, including events at random.

In [ ]:
rand_shuffle = np.random.choice(len(dataset), len(dataset), replace = False)
name = ISOTOPE + '_size' + str(sample_size)

# 20-20 marking for test and validation
test_split = int(len(dataset) * .2)
val_split = int(len(dataset) * .4)

test = dataset[rand_shuffle[:test_split],:,:]
val = dataset[rand_shuffle[test_split:val_split],:,:]
train = dataset[rand_shuffle[val_split:],:,:]
print(len(dataset))
print(test.shape, val.shape, train.shape)

os.makedirs('../data_splits/', exist_ok=True)
np.save('../data_splits/' + ISOTOPE + '_size' + str(sample_size)+'_test', test)
np.save('../data_splits/' + ISOTOPE + '_size' + str(sample_size)+'_val', val)
np.save('../data_splits/' + ISOTOPE + '_size' + str(sample_size)+'_train', train)
assert len(np.unique(np.isnan(train[:,:,4]))) == 1, 'NaNs in dataset'
assert len(np.unique(np.isnan(val[:,:,4]))) == 1, 'NaNs in dataset'
assert len(np.unique(np.isnan(test[:,:,4]))) == 1, 'NaNs in dataset'

# Make Histogram

In [ ]:
# Make histogram of values (either x,y,z,charge) from selected npy (train, val, test)
# Will need to change PLOT and DATA_SET_NAME to plot X-Y-Z-Q(charge) from training, val, or test
if dimension == 4:
    indices = ['X', 'Y', 'Z', 'A', 'ID', 'n/a', 'len']
else:
    indices = ['X', 'Y', 'Z', 'ID', 'n/a', 'len']
    
PLOT = 'ID'
DATA_SET_NAME = '_train'
index = indices.index(PLOT)

ds = np.load('../data_splits/' + ISOTOPE + '_size' + str(sample_size) + DATA_SET_NAME + '.npy')
if index >= dimension:
    info = ds[:,0,index].flatten()
else:
    info = ds[:,:,index].flatten()
plt.hist(info, bins=100)
plt.ylabel('probability')
plt.xlabel('value')
plt.title(PLOT + ' Distributions')
plt.show()

# Voxelation Process
Normalizing coords into Unit Cube This section takes the maximum and minimum data points from the original data file and normalizes it to fit the whole event in a unit cube. It writes the normalized data to a data file ending in sampled_normal.np

In [ ]:
name = ISOTOPE + '_size' + str(sample_size)
data = np.load('../voxel_data/' + name + '.npy')

RANGES = {
            'MIN_X': -270.0,
            'MAX_X': 270.0,
            'MIN_Y': -270.0,
            'MAX_Y': 270.0,
            'MIN_Z': -185.0,
            'MAX_Z': 1155.0,
            'MIN_LOG_A': 0.0,
            'MAX_LOG_A': 8.60
        }


print("Maximums and minimums from the original data set:")
print(np.amax(data[:,:,0]), np.amax(data[:,:,1]), np.amax(data[:,:,2]))
print(np.amin(data[:,:,0]), np.amin(data[:,:,1]), np.amin(data[:,:,2]))

data[:,:,0] = data[:,:,0]+ abs(RANGES['MIN_X']) # WORKINGGGGBDFGDF ON IT 
data[:,:,0] = data[:,:,0]/ (RANGES['MAX_X'] + abs(RANGES['MIN_X'])) 
data[:,:,1] = data[:,:,1]+ abs(RANGES['MIN_Y'])
data[:,:,1] = data[:,:,1]/ (RANGES['MAX_Y'] + abs(RANGES['MIN_Y']))
data[:,:,2] = data[:,:,2]+ abs(RANGES['MIN_Z'])
data[:,:,2] = data[:,:,2]/ (RANGES['MAX_Z'] + abs(RANGES['MIN_Z']))

print()
print("Normalized maximums and minimums:")
print(np.amax(data[:,:,0]), np.amax(data[:,:,1]), np.amax(data[:,:,2]))
print(np.amin(data[:,:,0]), np.amin(data[:,:,1]), np.amin(data[:,:,2]))

new_array_name = ISOTOPE + '_size' + str(sample_size) + '_sampled_normal'
np.save('../voxel_data/' + new_array_name, data)

The following two code blocks print the data as a histogram and uses assertion statements to make sure that we are normalizing everything correctly. In other words, we want to make sure we didn't change any of the points relative to each other and also must ensure that no points lie outside of the unit cube bounds.

In [ ]:
print(data[:,:,0].flatten())
plt.hist(data[:,:,0].flatten(), bins=100)
plt.xlabel('value')
plt.title('X Distributions')
plt.show()

plt.hist(data[:,:,1].flatten(), bins=100)
plt.xlabel('value')
plt.title('Y Distributions')
plt.show()

plt.hist(data[:,:,2].flatten(), bins=100)
plt.xlabel('value')
plt.title('Z Distributions')
plt.show()

### Assertion statements to check if coords are within unit cube

In [ ]:
name = ISOTOPE + '_size' + str(sample_size) + '_sampled_normal'
data = np.load('../voxel_data/' + name + '.npy')

assert np.amin(data[:,:,0]) > 0, 'There are x-values that are negative'
assert np.amin(data[:,:,1]) > 0, 'There are y-values that are negative'
assert np.amin(data[:,:,2]) > 0, 'There are z-values that are negative'
assert np.amax(data[:,:,0]) < 1, 'There are x-values that are greater than 1'
assert np.amax(data[:,:,1]) < 1, 'There are y-values that are greater than 1'
assert np.amax(data[:,:,2]) < 1, 'There are z-values that are greater than 1'

# Voxelizing and Labeling
This section breaks the data into K voxels and uses a dictionary to create and store labels for each voxel. It also outputs two files: one that ends in _base_voxels.npy and another that ends in _voxelated.npy. The _base_voxels.npy file has all of the voxels plotted on top of each other and the _voxelated.npy is the original data with all of the points being assigned a voxel_id. For more information on these two files, see the voxel-data-exploration.ipynb notebook.

In [ ]:
name = ISOTOPE + '_size' + str(sample_size) + '_sampled_normal' # Using normalized data
data = np.load('../voxel_data/' + name + '.npy')

K_x = 2
K_y = 2
K_z = 6

new_data = np.zeros((len(data), sample_size, 6), float)

#Store voxel bounds in dict with keys being lists
#Structure: voxel[key] = [ [voxel_lower_bounds], [voxel_upper_bounds], voxel_id]
voxels = dict({})
i=0

for z in range(K_z):
    for y in range(K_y):
        for x in range(K_x):
            key = [x,y,z]
            value = []
            #Creating lower bound of voxel
            min_bounds = [-1,-1,-1]
            min_bounds[0] = (1/K_x)*x
            min_bounds[1] = (1/K_y)*y
            min_bounds[2] = (1/K_z)*z
            #Creating upper bound of voxel
            max_bounds = [-2,-2,-2]
            max_bounds[0] = (1/K_x) + (1/K_x)*x
            max_bounds[1] = (1/K_y) + (1/K_y)*y
            max_bounds[2] = (1/K_z) + (1/K_z)*z
            
            value.append(min_bounds)
            value.append(max_bounds)
            value.append(i)
            i += 1
            
            voxels[str(key)] = value

# Identifying voxel id for each point and normalizing all points to be within [1/K_x,1/K_y,1/K_z]
# each instance will index according to the following 
# 0-x, 1-y, 2-z, 3-voxel id, 4-n/a (previously number of tracks), 5-event #

#indices: x, y, z, amplitude - 3, event index - 4, n/a (previously number of tracks) - 5, event length - 6

for i in tqdm.tqdm(range(len(data))):
    for j in range(sample_size):

        #Finding Current Point's Voxel Key
        voxel_key = [-1,-1,-1]
        x_val = data[i,j,0]
        if x_val < (1/K_x):
            voxel_key[0] = 0
        else:
            voxel_key[0] = 1

        y_val = data[i,j,1]
        if y_val < (1/K_y):
            voxel_key[1] = 0
        else:
            voxel_key[1] = 1

        z_val = data[i,j,2]
        voxel_key[2] = int(np.floor(z_val*K_z))

        #Getting related voxel info
        lower_bound = voxels[str(voxel_key)][0]
        upper_bound = voxels[str(voxel_key)][1]
        voxel_num = voxels[str(voxel_key)][2]

        #Normalizing coords
        new_x = x_val-lower_bound[0]
        new_y = y_val-lower_bound[1]
        new_z = z_val-lower_bound[2]
        
        #Saving new voxel coords and id
        new_data[i,j,0] = new_x
        new_data[i,j,1] = new_y
        new_data[i,j,2] = new_z
        data[i,j,3] = voxel_num
        new_data[i,j,3] = voxel_num
        data[i,j,4] = data[i,j,5]
        new_data[i,j,4] = data[i,j,4]
        data[i,j,5] = str(i)
        new_data[i,j,5] = str(i)
        
        # 0-x, 1-y, 2-z, 3-voxel id, 4-zeroed out (previously number of tracks), 5-event #

        
        # before: x, y, z, amplitude - 3, event index - 4, zeroed out (previously label) - 5, event length - 6

        #indices: x, y, z, voxel id - 3, amplitude - 4, zeroed out (previously label) - 5, event length - 6
        
# new_data[:,:,0] represents the x values, new_data[:,:,1] is y, and so on. So the cell prints the maxs and mins of x,y, and z values
print(np.amax(new_data[:,:,0]), np.amax(new_data[:,:,1]), np.amax(new_data[:,:,2]))
print(np.amin(new_data[:,:,0]), np.amin(new_data[:,:,1]), np.amin(new_data[:,:,2]))
  
# Converting all voxel keys from list to corresponding voxel id
i = 0

for z in range(K_z):
    for y in range(K_y):
        for x in range(K_x):
            key = [x,y,z]
            voxels[i] = voxels[str(key)]
            del voxels[str(key)]
            i += 1
            
new_array_name1 = ISOTOPE + '_size' + str(sample_size) + '_base_voxels.npy'
new_array_name2 = ISOTOPE + '_size' + str(sample_size) + '_voxelated.npy'

np.save('../voxel_data/' + new_array_name1, new_data)
np.save('../voxel_data/' + new_array_name2, data[:,:,:6]) # THIS is the file to use for incorporating unshuffled data in training set.

In [ ]:
voxels_np = np.zeros((K_x * K_y * K_z,2,3))
for i in range(K_x * K_y * K_z):
    min_bounds = voxels[i][0]
    max_bounds = voxels[i][1]
    voxels_np[i,0] = min_bounds
    voxels_np[i,1] = max_bounds

# print(voxels)
# print(voxels_np)
np.save('../voxel_data/voxel_bounds.npy', voxels_np)

In [ ]:
name1 = 'O16' + '_size' + str(512) + '_voxelated'
name2 = 'O16' + '_size' + str(512) + '_base_voxels'
voxel_data = np.load('../voxel_data/' + name1 + '.npy')
next_step_data = np.load('../voxel_data/' + name2 + '.npy')

print(voxel_data.shape, next_step_data.shape)

assert voxel_data.shape == (count, sample_size, 6), 'Voxelated Shape is incorrect'
assert next_step_data.shape == (count, sample_size, 6), 'Base Voxels Shape is incorrect'

# Shuffling and Randomly Augmenting Voxels

In [ ]:
name = ISOTOPE + '_size' + str(sample_size) + '_base_voxels'
name_unshuffled = ISOTOPE + '_size' + str(sample_size) + '_voxelated'
data = np.load('../voxel_data/' + name + '.npy')           # This is the data file with all of the voxels plotted on each other in one voxel. This makes the shuffling process easier
data_unshuffled = np.load('../voxel_data/' + name_unshuffled + '.npy')           # This is the original data with the normalized, voxelized event

new_data = np.zeros((len(data), sample_size, 6), float)
for i in tqdm.tqdm(range(len(data))):
    
    #Gets a list of where each voxel is going to be shuffled
    #Repeats until each voxel is assigned an id other than its own
    flag = True
    while flag:
        permutations = []
        ids = []
        overlap = False
        for j in range(K_x * K_y * K_z):
            ids.append(j)
        for j in range(K_x * K_y * K_z):
            val = random.choice(ids)
            permutations.append(val)
            ids.remove(val)
        for j,x in enumerate(permutations):
            if j == x:
                overlap = True
        if overlap == False:
            flag = False
            
    #Moves each point to its new voxel
    for j in range(sample_size):
        augment = 0
        #Optional random augmentaion (slightly changing xyz) for better generalization
        # rand = random.randint(1, 100)
        # if rand < 17:
            #Maximum point can shift is 1/20 of a unit cube
            #augment = (random.random())/20
            #if (rand%2) == 0:
                #augment = augment*-1
                  
        old_id = data[i,j,3]
        num_tracks = data[i,j,4]
        event_num = data[i,j,5]
        
        new_id = permutations[int(old_id)]
        new_min_bounds = voxels[new_id][0]
        new_x = data[i,j,0] + new_min_bounds[0] + augment
        new_y = data[i,j,1] + new_min_bounds[1] + augment
        new_z = data[i,j,2] + new_min_bounds[2] + augment
        
        #Doesn't include augment if causes point to move out of unit cube
        if (new_x < 0) or (new_x > 1):
            new_x = data[i,j,0] + new_min_bounds[0]
        if (new_y < 0) or (new_y > 1):
            new_y = data[i,j,1] + new_min_bounds[1]
        if (new_z < 0) or (new_z > 1):
            new_z = data[i,j,2] + new_min_bounds[2]

        new_data[i,j,0] = new_x
        new_data[i,j,1] = new_y
        new_data[i,j,2] = new_z
        new_data[i,j,3] = old_id
        new_data[i,j,4] = num_tracks # zeroed out
        new_data[i,j,5] = event_num
        
final_data = np.concatenate((data_unshuffled[:1000],new_data),axis=0) 
# Use the above line to change the proportions of unshuffled and shuffled data in the training set

print(final_data.shape)
# to confirm concatenation occured correctly

shuffled_data = ISOTOPE + '_size' + str(sample_size) + '_shuffled_voxels_only' # Only shuffled
np.save('../voxel_data/' + shuffled_data, new_data)

new_array_name = ISOTOPE + '_size' + str(sample_size) + '_shuffled_voxels' # Includes unshuffled
np.save('../voxel_data/' + new_array_name, final_data)

### Assertion statements to check if coords are still within unit cube

In [ ]:
name = ISOTOPE + '_size' + str(512) + '_shuffled_voxels'
data = np.load('../voxel_data/' + name + '.npy')

assert np.amin(data[:,:,0]) >= 0, 'There are x-values that are negative'
assert np.amin(data[:,:,1]) >= 0, 'There are y-values that are negative'
assert np.amin(data[:,:,2]) > 0, 'There are z-values that are negative'
assert np.amax(data[:,:,0]) < 1, 'There are x-values that are greater than 1'
assert np.amax(data[:,:,1]) < 1, 'There are y-values that are greater than 1'
assert np.amax(data[:,:,2]) < 1, 'There are z-values that are greater than 1'

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Extract the x, y, z coordinates from event_data
x = new_data[12,:, 0]
y = new_data[12,:, 1]
z = new_data[12,:, 2]

ax.scatter(x, z, y, s=1)
#ax.set_xlim(-250,250)
#ax.set_ylim(0,1000)
#ax.set_zlim(-250,250)
ax.set_xlabel('X [mm]')
ax.set_ylabel('Y [mm]')
ax.set_zlabel('Z [mm]')
plt.title('3D Scatter Plot of Event Data')

plt.show()

# Split Testing Set, Training Set, and Validation Set

In [ ]:
# performs a 20-test 20-val 60-train split on all 4-track events
# generates an array of numbers as long as the length of the data to randomize the events 
name = ISOTOPE + '_size' + str(sample_size)
all_events = np.load('../voxel_data/' + name + '_shuffled_voxels.npy')
rand_shuffle = np.random.choice(len(all_events), len(all_events), replace = False)


# 20-20 marking for test and validation
test_split = int(len(all_events) * .2)
val_split = int(len(all_events) * .4)


test_data =  all_events[rand_shuffle[:test_split],:,:]    #only saving the indices and number of tracks of the test events
val_data = all_events[rand_shuffle[test_split:val_split],:,:]
train_data = all_events[rand_shuffle[val_split:],:,:]


print(test_data.shape, val_data.shape, train_data.shape)
np.save('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'test', test_data)
np.save('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'train', train_data)
np.save('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'val', val_data)

# Making Test Sets

In [ ]:
prev_data = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + '_sampled.npy')
tr = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'train.npy')
va = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'val.npy')
te = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'test.npy')

#works perfect if length of dataset is even, if odd, rounding may be off
#assert tr.shape == (np.ceil(len(prev_data) * .6) , sample_size, 4), 'Array has incorrect shape, check first for rounding error.'
#assert va.shape == (np.ceil(len(prev_data) * .2) , sample_size, 4), 'Array has incorrect shape, check first for rounding error.'
#assert te.shape == (np.ceil(len(prev_data) * .2) , sample_size, 4), 'Array has incorrect shape, check first for rounding error.'
print(len(prev_data))
print(tr.shape, va.shape, te.shape)
print(len(np.unique(tr[:,:,3])))

## Checking Test, Training, and Validation

In [ ]:
check = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'train.npy')
check2 = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'val.npy')
check3 = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'test.npy')


print(check.shape)
print(check2.shape)
print(check3.shape)
print('')

print(check[1][0][:])
print('')

print(len(np.unique(check[:,:,3])))
print(len(np.unique(check2[:,:,3])))
print(len(np.unique(check3[:,:,3])))

# Checking data sets

In [ ]:
#load in test, val, and train np arrays
test = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'test.npy')
train = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'train.npy')
val = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'val.npy')

In [ ]:
#ASSERT
train = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'train.npy')
test = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'test.npy')
val = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'val.npy')

assert np.sum(np.isnan(train)) == 0, 'NaNs in dataset'
assert np.sum(np.isnan(test)) == 0, 'NaNs in dataset'
assert np.sum(np.isnan(val)) == 0, 'NaNs in dataset'
assert np.sum(np.isinf(train)) == 0, 'Infinities in dataset'
assert np.sum(np.isinf(test)) == 0, 'Infinities in dataset'
assert np.sum(np.isinf(val)) == 0, 'Infinities in dataset'

# Make Histogram

In [ ]:
CLASSIFICATION = 'Voxel'
class_type = CLASSIFICATION
PROJECTION = 'XYZ'
ISOTOPE = 'O16'

PROJ_TO_COLS = {'XYZ': [0,1,2]}

user_input = PROJ_TO_COLS[PROJECTION]
print(class_type)
print(user_input)

In [ ]:
# Make histogram of values (either x,y,z,charge) from selected npy (train, val, test)
# Will need to change PLOT and DATA_SET_NAME to plot X-Y-Z-Q(charge) from training, val, or test
PLOT = 'Z'
DATA_SET_NAME = 'train'
index = PROJECTION.find(PLOT)
print(index)

if index != -1: #So that it is either X, Y or Z (0,1 or 2 indices)
    data = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + DATA_SET_NAME + '.npy')
    info = data[:,:,index].flatten()
    plt.hist(info, density=True, bins=100)
    plt.ylabel('probability')
    plt.xlabel('value')
    plt.title(PLOT + ' Distributions')
    plt.show()
    # plt.savefig('data/'+ '.png', bbox_inches = 'tight') # uncomment to save
else:
    print('Value to plot is invalid, change PLOT')

In [ ]:
#plotting the x,y,and z distributions for training, validation, and test sets

CLASSIFICATION = 'MULTI-CLASS'
class_type = CLASSIFICATION
PROJECTION = 'XYZ'
ISOTOPE = 'O16'
PLOT = ['X', 'Y','Z']
DATA_SET_NAME = ['train','val','test']
for i in DATA_SET_NAME:
    for j in PLOT:
        index = PROJECTION.find(j)
        if index != -1:
            data = np.load('../voxel_data/' +ISOTOPE+ '_size'+ str(sample_size) +i+ '.npy')
            info = data[:,:,index].flatten()
            plt.hist(info, density=True, bins=100)
            plt.ylabel('probability')
            plt.xlabel('value')
            plt.title(j + ' Distributions in ' + i)
            plt.show()
            #plt.savefig('data/' +i+j+ '.png', bbox_inches = 'tight') 
        else:
            print('Value to plot is invalid, change PLOT')#

In [ ]:
name = ISOTOPE + '_size' + str(sample_size) + '_voxelated'
data = np.load('../voxel_data/' + name + '.npy')

for i in range(len(data)):
    for j in range(512):
        if data[i,j,4] == 5:
            print(data[i,j,5])
print(ISOTOPE)

In [ ]:
test = np.load('../voxel_data/' + ISOTOPE + '_size' + str(sample_size) + 'test.npy')

event_nums = np.zeros((len(test),))
for i in range(len(test)):
    event_nums[i] = test[i,0,5]